# Import Libraries

In [1]:
import os
import random
import shutil
import numpy as np
import pandas as pd
from tqdm import tqdm

import tensorflow as tf
from tensorflow.keras import layers, models, Model
from tensorflow.keras.applications.mobilenet_v2 import MobileNetV2, preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

import matplotlib.pyplot as plt
import cv2

2026-04-30 05:39:26.523301: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1777527566.746515      57 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1777527566.811948      57 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1777527567.351748      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777527567.351845      57 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1777527567.351849      57 computation_placer.cc:177] computation placer alr

# Input Files

In [6]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    print(dirname)

/kaggle/input
/kaggle/input/datasets
/kaggle/input/datasets/paultimothymooney
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val/PNEUMONIA
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/val/NORMAL
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test/PNEUMONIA
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/test/NORMAL
/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray/train
/kaggle/input/datasets/paultimothymooney/chest-x

In [2]:
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename)

SyntaxError: incomplete input (4145440086.py, line 3)

In [7]:
base_dir = "/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray"

train_dir = os.path.join(base_dir, "train")
val_dir   = os.path.join(base_dir, "val")
test_dir  = os.path.join(base_dir, "test")

In [8]:
test_dir

'/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/test'

In [9]:
IMG_SIZE = (224, 224)
BATCH_SIZE = 16

datagen = ImageDataGenerator(
    rescale=1./255
)

train_gen = datagen.flow_from_directory(
    train_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

val_gen = datagen.flow_from_directory(
    val_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary'
)

test_gen = datagen.flow_from_directory(
    test_dir,
    target_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    shuffle=False
)

Found 5216 images belonging to 2 classes.
Found 16 images belonging to 2 classes.
Found 624 images belonging to 2 classes.


# Create Balance Dataset
- Train: 80%
- Test: 20%

In [ ]:
def create_balanced_subset(source_dir, target_dir, images_per_class=600):
    import os, shutil, random

    classes = ['NORMAL', 'PNEUMONIA']

    if os.path.exists(target_dir):
        shutil.rmtree(target_dir)

    for split in ['train', 'test']:
        for cls in classes:
            os.makedirs(os.path.join(target_dir, split, cls), exist_ok=True)

    for cls in classes:
        pool = []

        for split in ['train', 'test', 'val']:
            path = os.path.join(source_dir, split, cls)

            if os.path.exists(path):
                for f in os.listdir(path):
                    if f.startswith('.'):
                        continue

                    if f.lower().endswith(('.jpeg', '.jpg', '.png')):
                        pool.append(os.path.join(path, f))

        print(f"{cls} total images found:", len(pool))

        random.seed(42)
        random.shuffle(pool)

        subset = pool[:images_per_class]

        split_idx = int(len(subset) * 0.8)
        train_imgs = subset[:split_idx]
        test_imgs = subset[split_idx:]

        for img in train_imgs:
            shutil.copy(img, os.path.join(target_dir, 'train', cls))

        for img in test_imgs:
            shutil.copy(img, os.path.join(target_dir, 'test', cls))

    print("Dataset created successfully!")

In [ ]:
target_path = '/kaggle/working/reduced_pneumonia_data'

create_balanced_subset(
    source_dir='/kaggle/input/datasets/paultimothymooney/chest-xray-pneumonia/chest_xray/chest_xray',
    target_dir=target_path,
    images_per_class=600
)

In [ ]:
for cls in ['NORMAL', 'PNEUMONIA']:
    path = f'/kaggle/working/reduced_pneumonia_data/train/{cls}'
    print(cls, ":", len(os.listdir(path)))

# Build Model

In [ ]:
def build_model(input_shape=(224, 224, 3)):
    
    base_model = tf.keras.applications.MobileNetV2(
        weights='imagenet',
        include_top=False,
        input_shape=input_shape
    )

    base_model.trainable = False

    inputs = tf.keras.Input(shape=input_shape)

    x = base_model(inputs, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dense(128, activation='relu')(x)
    x = layers.Dropout(0.3)(x)
    outputs = layers.Dense(1, activation='sigmoid')(x)

    model = tf.keras.Model(inputs, outputs)

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy',
                 tf.keras.metrics.Precision(),
                 tf.keras.metrics.Recall()]
    )

    return model

model = build_model()
model.summary()

# Data Generator

In [ ]:
# 1. Data Augmentation for the reduced dataset
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True,
    validation_split=0.2 # Using 20% of the subset for validation
)

# 2. Loaders
train_generator = train_datagen.flow_from_directory(
    '/kaggle/working/reduced_pneumonia_data/train',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
    subset='training'
)

validation_generator = train_datagen.flow_from_directory(
    '/kaggle/working/reduced_pneumonia_data/train',
    target_size=(224, 224),
    batch_size=16,
    class_mode='binary',
    subset='validation'
)

# Model Training

In [ ]:
history = model.fit(
    train_generator,
    validation_data=validation_generator,
    epochs=5
)

# 4. Save the model for your Streamlit UI later
model.save('pneumonia_model_mobilenet.h5')
print("Model saved successfully!")

# Model Validation

In [ ]:
print("Training Accuracy:", history.history['accuracy'][-1])
print("Validation Accuracy:", history.history['val_accuracy'][-1])

In [ ]:
results = model.evaluate(validation_generator)

for name, value in zip(model.metrics_names, results):
    print(name, ":", value)

# GRAD-CAM

In [ ]:
import numpy as np
import cv2
import matplotlib.pyplot as plt

def make_gradcam_heatmap(img_array, model, last_conv_layer_name, pred_index=None):
    # First, we create a model that maps the input image to the activations
    # of the last conv layer as well as the output predictions
    grad_model = tf.keras.models.Model(
        [model.inputs], [model.get_layer('mobilenetv2_1.00_224').get_layer(last_conv_layer_name).output, model.output]
    )

    with tf.GradientTape() as tape:
        last_conv_layer_output, preds = grad_model(img_array)
        if pred_index is None:
            pred_index = tf.argmax(preds[0])
        class_channel = preds[:, pred_index]

    # This is the gradient of the output node with regard to the output feature map
    grads = tape.gradient(class_channel, last_conv_layer_output)

    # This is a vector where each entry is the mean intensity of the gradient over a specific feature map channel
    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # We multiply each channel in the feature map array by "how important this channel is"
    last_conv_layer_output = last_conv_layer_output[0]
    heatmap = last_conv_layer_output @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    # Normalize the heatmap between 0 & 1
    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

In [ ]:
import tensorflow as tf
import numpy as np

def make_gradcam_heatmap(img_array, model, last_conv_layer_name="out_relu"):
    grad_model = tf.keras.models.Model(
        inputs=model.input,
        outputs=[
            model.get_layer(last_conv_layer_name).output,
            model.output
        ]
    )

    with tf.GradientTape() as tape:
        conv_outputs, predictions = grad_model(img_array)
        loss = predictions[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = conv_outputs @ pooled_grads[..., tf.newaxis]
    heatmap = tf.squeeze(heatmap)

    heatmap = tf.maximum(heatmap, 0) / tf.math.reduce_max(heatmap)
    return heatmap.numpy()

In [ ]:
import numpy as np
import tensorflow as tf

def make_gradcam_heatmap(img_array):

    img_tensor = tf.convert_to_tensor(img_array, dtype=tf.float32)

    with tf.GradientTape() as tape:
        conv_outputs, preds = grad_model(img_tensor, training=False)
        loss = preds[:, 0]

    grads = tape.gradient(loss, conv_outputs)

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    conv_outputs = conv_outputs[0]

    heatmap = tf.reduce_sum(conv_outputs * pooled_grads, axis=-1)

    heatmap = tf.maximum(heatmap, 0)
    heatmap /= (tf.reduce_max(heatmap) + 1e-8)

    return heatmap.numpy()

In [ ]:
def run_visualization(img_path, model):

    img = tf.keras.preprocessing.image.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) / 255.0
    
    heatmap = make_gradcam_heatmap(img_array)
    
    # run_visualization(img_path, model)

    result_img = display_gradcam(img_path, heatmap)

    plt.figure(figsize=(10,5))

    plt.subplot(1,2,1)
    plt.title("Original X-Ray")
    plt.imshow(img)
    plt.axis("off")

    plt.subplot(1,2,2)
    plt.title("Grad-CAM")
    plt.imshow(result_img)
    plt.axis("off")

    plt.show()

In [ ]:
img_path = "/kaggle/working/reduced_pneumonia_data/test/PNEUMONIA/person791_virus_1422.jpeg"
run_visualization(img_path, model)

In [ ]:
import os

base = "/kaggle/working/reduced_pneumonia_data/test/PNEUMONIA"

print(os.listdir(base)[:5])

In [ ]:
from tensorflow.keras.preprocessing import image
image.load_img(img_path)